# Tempo Run 2025 — Enrich data bằng Qwen3 API

**Tuỳ chọn**: Bỏ qua nếu không muốn dùng API hoặc đã hết quota.
Mặc định chỉ paraphrase (giữ đáp án gốc). Dùng `--synth` để sinh thêm câu hỏi mới (tốn thêm token).

**Cảnh báo quota**: 1M token free. Paraphrase ~3000 mẫu có thể tốn ~600K-1M token. Bắt đầu với `--synth-max 0` để tiết kiệm.

In [ ]:
%cd /content/finetune_1B_MCQ_VN
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os
assert os.environ.get("DASHSCOPE_API_KEY"), "DASHSCOPE_API_KEY missing in .env"

In [ ]:
# 1. Smoke test API (1 call)
from temprun.enrich import call_chat, get_client, get_model, get_model_source
client = get_client()
model = get_model()
out = call_chat(client, [
    {"role": "system", "content": "Bạn là trợ lý hữu ích."},
    {"role": "user", "content": "Trả lời ngắn gọn: 2+2=?"},
], max_tokens=20, temperature=0.0)
print('model:', model, '| source:', get_model_source())
print('output:', out)

In [ ]:
# 2. Chạy enrichment thực sự (idempotent — chạy lại không tốn thêm token cho row đã xử lý)
# Mặc định: paraphrase + explain. Không sinh Q mới.
# Script sẽ in: model đang dùng + heartbeat progress mỗi 25 hàng (sync)
# hoặc 'starting/done' lines mỗi phase (async). Đổi tần suất bằng --progress-every N.
# Thêm --push-after để backup enriched.jsonl + cache lên HF sau khi xong
# (tránh mất tiền API khi Colab reset). Cần HF_TOKEN write scope trong .env.
!python scripts/enrich_data.py \
    --in  data/processed/train.jsonl \
    --out data/processed/enriched.jsonl

In [ ]:
# 3. (Tuỳ chọn) Thêm sinh câu hỏi mới — tốn thêm ~200K token
# Chỉ chạy nếu còn quota. Bỏ comment nếu muốn.

# !python scripts/enrich_data.py \
#     --in  data/processed/enriched.jsonl \
#     --out data/processed/enriched.jsonl \
#     --no-paraphrase --no-explain --synth --synth-max 200

In [ ]:
# 4. Merge enriched → final/{train,eval}.jsonl (90/10 split lại)
!python scripts/merge_enriched.py

In [ ]:
# 5. (Tuỳ chọn) Backup enriched + cache lên HF (cùng repo zip, vào processed/).
#    Chỉ cần khi bạn sợ mất Colab session. Tránh phải chạy lại API call.
!python scripts/enrich_data.py \
    --in  data/processed/enriched.jsonl \
    --out data/processed/enriched.jsonl \
    --no-paraphrase --no-explain --push-after


In [ ]:
# 6. Verify final size
!wc -l data/processed/final/*.jsonl
!head -1 data/processed/final/train.jsonl | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('label:', r['label']); print('source:', r.get('_source', 'original'))"

## (Tuỳ chọn) Restore enriched + cache trên session mới

Nếu bạn đã push lên HF ở session trước (qua `--push-after`), chạy cell dưới đây ở
session mới **trước khi** chạy `enrich_data.py` để khôi phục `enriched.jsonl` + cache.
Sau đó `enrich_data.py` chạy idempotent (đọc cache, không tốn thêm API cho row đã xong).

Cell này tải về `data/processed/_hf_cache/processed/enriched.jsonl` rồi `shutil.move` về
`data/processed/enriched.jsonl` (đúng vị trí mà `enrich_data.py` và `merge_enriched.py` mong đợi).

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os

REPO = os.environ["HF_DATASET_REPO"]
TOKEN = os.environ["HF_TOKEN"]
target = repo_root() / "data" / "processed"
for fname in ("enriched.jsonl", "enriched.jsonl.cache.json"):
    in_repo = f"processed/{fname}"
    downloaded = Path(hf_hub_download(
        repo_id=REPO, filename=in_repo, repo_type="dataset", token=TOKEN,
        local_dir=target / "_hf_cache",
    ))
    final = target / fname
    if final.exists():
        final.unlink()
    shutil.move(str(downloaded), str(final))
    print(f"restored: {final}")
